## This script is to convert recorded demonstrations (from rosbag format) to samples (in txt format) to conform with the TP-GMM code.

In [ ]:
task_name = "pick"
nbFrames = 2

In [ ]:
import rosbag
import numpy as np
import matplotlib.pyplot as plt
from math import sqrt
## System and directories stuff
import sys
import os
from pathlib import Path
import pickle


## This is important to add because if run from 'tp_gmm.py', 
## ros changes the working directory by default to ../.ros/..
os.chdir("/home/erl/Multicobot-UR10/src")
WS_DIR = Path.cwd()
print(WS_DIR)

tasks_dir = WS_DIR / "tp_gmm/tasks"
# # Getting the task_name from tp-gmm.py
# with open(tasks_dir / 'task_name.pkl', 'rb') as fp:
#     task_name_file = pickle.load(fp)
#     print(task_name_file)
# task_name = task_name_file['task_name']

demons_dir = str(WS_DIR / f'motion_planning/bag_files/Trajectory_Data_Collection/Demons/{task_name}') + '/'
data_dir = str(WS_DIR / f'tp_gmm/data/{task_name}') + '/'

print(demons_dir)
print(data_dir)


### Defining the directory according to the given task_name, and thus loop over the Demons in the directory (task_name, nbDemons in dir, ref_demon)

In [ ]:
demons_names = []
demons_nums = []
for filename in os.listdir(demons_dir):
    demons_names.append(filename)
    demons_nums.append(filename[filename.find('demon'):filename.find('.bag')])

print(demons_names)
print(demons_nums)

nbdemons = len(demons_names)
print("Number of Demons: ", nbdemons)

### 3D

In [ ]:
from geometry_msgs.msg import Pose, PoseArray
from copy import deepcopy
from scipy.spatial.transform import Rotation as R


# ref_demon = {'ref':'', 'ref_nbpoints': 100000, 'nbDemons': nbdemons, 'demons_nums': demons_nums} # Choosing the shortest demon to be the reference in DTW
ref_demon = {'ref':'', 'ref_nbpoints': 0, 'nbDemons': nbdemons, 'demons_nums': demons_nums, 'nbFrames': nbFrames} # Choosing the longest demon to be the reference in DTW

for d in range(nbdemons):

    demon_bag = rosbag.Bag(demons_dir + demons_names[d], 'r')

    posearray_topic = "/ur10_1/planned_trajectory/posearray"
    start_pose_topic = "/ur10_1/start_pose"
    goal_pose_topic = "/ur10_1/goal_pose"
    
    for topic, msg, t in demon_bag.read_messages(topics=[start_pose_topic, goal_pose_topic]):
        # print(topic)
        if topic == start_pose_topic:
            pos = [msg.pose.position.x, msg.pose.position.y, msg.pose.position.z]
            orient = R.from_quat([msg.pose.orientation.x, msg.pose.orientation.y, msg.pose.orientation.z, msg.pose.orientation.w])
            # print(orient.as_quat())
            # print(orient.as_euler('ZYX'))
            # print(orient.as_matrix())

            arr_frame1_b = np.array([[0], [pos[0]], [pos[1]], [pos[2]]])
            arr_frame1_A = np.eye(4)
            arr_frame1_A[1:,1:] = orient.as_matrix()       
            # print(arr_frame1_A)  

        elif topic == goal_pose_topic:
            pos = [msg.pose.position.x, msg.pose.position.y, msg.pose.position.z]
            orient = R.from_quat([msg.pose.orientation.x, msg.pose.orientation.y, msg.pose.orientation.z, msg.pose.orientation.w])
            # print(orient.as_quat())
            # print(orient.as_euler('ZYX'))
            # print(orient.as_matrix())

            arr_frame2_b = np.array([[0], [pos[0]], [pos[1]], [pos[2]]])
            arr_frame2_A = np.eye(4)
            arr_frame2_A[1:,1:] = orient.as_matrix()
            # print(arr_frame2_A)

    dim = 1 + 3
    conc_arr_Data = np.zeros((dim,1))
    down_sample = 1 # 30
    # msg_count = 0
    for topic, msg, t in demon_bag.read_messages(topics=[posearray_topic]):
        for pose_count, pose in enumerate(msg.poses):
            # Preparing the Data points matrix from the Demons
            # #######################-----------------------------------------------

            # ## For transforming the pose from (_ee_link) to (flange_gmm) w.r.t _base_link_gmm
            # ## NOTE: This is because the end_effector frame used in 'collect_trajectories_data.py' is '_ee_link', so we have to transform it to _flange_gmm to be used later on as 
            # ## _flange_gmm is the frame which Task Parameters of the TP-GMM are defined with respect to it. Task Parameters are defined to have the base link (_base_link_gmm) and end-effector link (_flange_gmm)
            # ee_link_to_flange_gmm_posi = np.array([0.000, 0.000, 0.000])
            # ee_link_to_flange_gmm_ori = R.from_quat([0.7071068, 0.0, 0.7071068, 0.0])

            # base_gmm_to_ee_posi = np.array([pose.position.x, pose.position.y, pose.position.z])
            # base_gmm_to_ee_ori = R.from_quat([pose.orientation.x, pose.orientation.y, pose.orientation.z, pose.orientation.w])

            # base_gmm_to_flange_gmm_posi = base_gmm_to_ee_posi + base_gmm_to_ee_ori.apply(ee_link_to_flange_gmm_posi)
            # base_gmm_to_flange_gmm_ori = base_gmm_to_ee_ori * ee_link_to_flange_gmm_ori
            
            # #######################-----------------------------------------------
        
            # arr_Data = np.array([[sqrt(msg.O_T_EE_c[12]**2 + msg.O_T_EE_c[13]**2 + msg.O_T_EE_c[14]**2)], [msg.O_T_EE[12]], [msg.O_T_EE[13]], [msg.O_T_EE[14]]]) # Euc. distance of error_pose
            arr_Data = np.array([[pose_count], [pose.position.x], [pose.position.y], [pose.position.z]]) # time sample
            # arr_Data = np.array([[pose_count], [base_gmm_to_flange_gmm_posi[0]], [base_gmm_to_flange_gmm_posi[1]], [base_gmm_to_flange_gmm_posi[2]]]) # time sample
            conc_arr_Data = np.concatenate((conc_arr_Data, arr_Data), axis=1)

            # # Preparing the b matrix & A matrix for frame 1 (Position & RotationMatrix of the Starting point(frame) of the Demon w.r.t the base frame of the robot (panda_link0))
            # if (msg_count == 0):
            #     arr_frame1_b = np.array([[0], [msg.O_T_EE[12]], [msg.O_T_EE[13]], [msg.O_T_EE[14]]])
            #     arr_frame1_A = np.matrix([[1,             0,             0,             0],
            #                             [0, msg.O_T_EE[0], msg.O_T_EE[4], msg.O_T_EE[8]],
            #                             [0, msg.O_T_EE[1], msg.O_T_EE[5], msg.O_T_EE[9]],
            #                             [0, msg.O_T_EE[2], msg.O_T_EE[6], msg.O_T_EE[10]]])

            # # Preparing the b matrix & A matrix for frame 2 (Position & RotationMatrix of the Ending point(frame) of the Demon w.r.t the base frame of the robot (panda_link0))
            # arr_frame2_b = np.array([[0], [msg.O_T_EE[12]], [msg.O_T_EE[13]], [msg.O_T_EE[14]]])
            # arr_frame2_A = np.matrix([[1,             0,             0,             0],
            #                         [0, msg.O_T_EE[0], msg.O_T_EE[4], msg.O_T_EE[8]],
            #                         [0, msg.O_T_EE[1], msg.O_T_EE[5], msg.O_T_EE[9]],
            #                         [0, msg.O_T_EE[2], msg.O_T_EE[6], msg.O_T_EE[10]]])
            
        # msg_count += 1
            # print(pose_count)
            # print(arr_Data.T)
            # print(pose)
    # print("---")
    
    # # debugging
    # if demons_nums[d] == "demon_27":
    #     print("Ref demon data points: ",  conc_arr_Data.shape)
    #     print(conc_arr_Data[:,12])
    #     print(pose_count)
    # #\ debugging
    
    demon_bag.close()
    
    print("Number of data points in {}: {}".format(demons_nums[d], conc_arr_Data.shape[1]-1)) #, msg_count)
    # Choosing the shortest Demon to be the reference Demon
    # if (msg_count < ref_demon['ref_nbpoints']):
    #     ref_demon["ref"] = demons_nums[d]
    #     ref_demon["ref_nbpoints"] = msg_count
    # if (conc_arr_Data.shape[1]-1 < ref_demon['ref_nbpoints']):
    #     ref_demon["ref"] = demons_nums[d]
    #     ref_demon["ref_nbpoints"] = conc_arr_Data.shape[1]-1 # because the DTW reduces 1 point!! # msg_count
    #     ref_demon["down_sample_factor"] = down_sample    
    # Choosing the longest Demon to be the reference Demon
    # if (msg_count > ref_demon['ref_nbpoints']):
    if (conc_arr_Data.shape[1]-1 > ref_demon['ref_nbpoints']):
        ref_demon["ref"] = demons_nums[d]
        ref_demon["ref_nbpoints"] = conc_arr_Data.shape[1]-1 # because the DTW reduces 1 point!! # msg_count
        ref_demon["down_sample_factor"] = down_sample  

    # Filling in b matrix & A matrix for frame 1
    conc_arr_frame1_b = np.ones((dim, pose_count)) * arr_frame1_b
    conc_arr_frame1_A = np.tile(arr_frame1_A, pose_count) # the .T is because the rotation is around negative y-axis

    # Filling in b matrix & A matrix for frame 2
    conc_arr_frame2_b = np.ones((dim, pose_count)) * arr_frame2_b
    conc_arr_frame2_A = np.tile(arr_frame2_A, pose_count)

    # Saving in txt files
    np.savetxt(data_dir + demons_nums[d] + "_sample_Data.txt", conc_arr_Data[:,1:], fmt='%.5f') # cut out the 1st column (zeros) that was created only to initialize the array 
    np.savetxt(data_dir + demons_nums[d] + "_sample_frame1_b.txt", conc_arr_frame1_b, fmt='%.5f')
    np.savetxt(data_dir + demons_nums[d] + "_sample_frame1_A.txt", conc_arr_frame1_A, fmt='%.5f')
    np.savetxt(data_dir + demons_nums[d] + "_sample_frame2_b.txt", conc_arr_frame2_b, fmt='%.5f')
    np.savetxt(data_dir + demons_nums[d] + "_sample_frame2_A.txt", conc_arr_frame2_A, fmt='%.5f')

    # Add the ',' to put file in the format that TP-GMM code accepts
    with open(r"{}".format(data_dir) + demons_nums[d]+ "_sample_Data.txt", 'r') as f: # The 'r' before the directory name is to open the file as read-only
        data = f.read(); data = data.replace(' ', ',')
    with open(r"{}".format(data_dir) + demons_nums[d]+ "_sample_Data.txt", 'w') as f:
        f.write(data)
        
    with open(r"{}".format(data_dir) + demons_nums[d] + "_sample_frame1_b.txt", 'r') as f:
        data = f.read(); data = data.replace(' ', ',')
    with open(r"{}".format(data_dir) + demons_nums[d] + "_sample_frame1_b.txt", 'w') as f:
        f.write(data)

    with open(r"{}".format(data_dir) + demons_nums[d] + "_sample_frame1_A.txt", 'r') as f:
        data = f.read(); data = data.replace(' ', ',')
    with open(r"{}".format(data_dir) + demons_nums[d] + "_sample_frame1_A.txt", 'w') as f:
        f.write(data)

    with open(r"{}".format(data_dir) + demons_nums[d] + "_sample_frame2_b.txt", 'r') as f:
        data = f.read(); data = data.replace(' ', ',')
    with open(r"{}".format(data_dir) + demons_nums[d] + "_sample_frame2_b.txt", 'w') as f:
        f.write(data)

    with open(r"{}".format(data_dir) + demons_nums[d] + "_sample_frame2_A.txt", 'r') as f:
        data = f.read(); data = data.replace(' ', ',')
    with open(r"{}".format(data_dir) + demons_nums[d]+ "_sample_frame2_A.txt", 'w') as f:
        f.write(data)

print("Reference Demon:")
print(ref_demon)
with open(tasks_dir / f'{task_name}/demons_info.pkl', 'wb') as fp:
    pickle.dump(ref_demon, fp)

In [ ]:
print(arr_frame1_b)
print(arr_frame1_A[1:,1:])
quat = R.from_matrix(arr_frame1_A[1:,1:])

print("-----")

print(quat.as_quat())
print(arr_frame2_b)
print(arr_frame2_A[1:,1:])
quat = R.from_matrix(arr_frame2_A[1:,1:])
print(quat.as_quat())

### DTW the demons

In [ ]:
%matplotlib inline
# %matplotlib qt
from dtw import *

print("Reference Demon:")
ref_arr_Data = np.loadtxt(data_dir + ref_demon['ref'] + "_sample_Data.txt", delimiter=',')
print(ref_arr_Data.shape)
demons_nums_dtw = []

for d in range(nbdemons):
    
    print("DTWinng..")
    temp_arr_Data = np.loadtxt(data_dir + demons_nums[d] + "_sample_Data.txt", delimiter=',')
    print(temp_arr_Data.shape)

    DTW = dtw(temp_arr_Data[1:,:].T, ref_arr_Data[1:,:].T)
    wq = warp(DTW, index_reference=False)
    print(wq.shape)
    print(warpArea(DTW))
    print(DTW.distance)
    print(DTW.normalizedDistance)

    # if (DTW.normalizedDistance >= 0.30): continue # Filter out the odd demons
    
    ## Plotting
    fig = plt.figure()
    ax = plt.axes(projection='3d')

    ax.plot(ref_arr_Data[1,:], ref_arr_Data[2,:], ref_arr_Data[3,:], label='ref')
    ax.plot(temp_arr_Data[1,:], temp_arr_Data[2,:], temp_arr_Data[3,:], label='{}'.format(demons_nums[d]))
    ax.plot(temp_arr_Data[1,wq], temp_arr_Data[2,wq], temp_arr_Data[3,wq], label='DTW {}'.format(demons_nums[d]))
    
    ax.legend()
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_zlabel('z')

    ## Saving in .txt
    temp_arr_Data = temp_arr_Data[:,wq]
    temp_arr_Data[0,:] = ref_arr_Data[0,:ref_arr_Data.shape[1]]#-1] # NOTE: It seems that with the update of 'dtw' package, this '-1' here has to be removed

    np.savetxt(data_dir + demons_nums[d] + "_sample_Data.txt", temp_arr_Data, fmt='%.5f')
    with open(r"{}".format(data_dir) + demons_nums[d] + "_sample_Data.txt", 'r') as f: # The 'r' before the directory name is to open the file as read-only
        data = f.read(); data = data.replace(' ', ',')
    with open(r"{}".format(data_dir) + demons_nums[d] + "_sample_Data.txt", 'w') as f:
        f.write(data)
    # Re-shaping the A & b matrices .txt to be of the same shape as the reference Demon (longest or shortest)
    temp_arr_frame1_b = np.loadtxt(data_dir + demons_nums[d] + "_sample_frame1_b.txt", delimiter=',')
    temp_arr_frame1_A = np.loadtxt(data_dir + demons_nums[d] + "_sample_frame1_A.txt", delimiter=',')
    temp_arr_frame2_b = np.loadtxt(data_dir + demons_nums[d] + "_sample_frame2_b.txt", delimiter=',')
    temp_arr_frame2_A = np.loadtxt(data_dir + demons_nums[d] + "_sample_frame2_A.txt", delimiter=',')
    
    temp_arr_frame1_b = np.ones((ref_arr_Data.shape[0], ref_arr_Data.shape[1])) * temp_arr_frame1_b[:,0].reshape(ref_arr_Data.shape[0], 1)
    temp_arr_frame1_A = np.tile(temp_arr_frame1_A[:4,:4], ref_arr_Data.shape[1])
    temp_arr_frame2_b = np.ones((ref_arr_Data.shape[0], ref_arr_Data.shape[1])) * temp_arr_frame2_b[:,0].reshape(ref_arr_Data.shape[0], 1)
    temp_arr_frame2_A = np.tile(temp_arr_frame2_A[:4,:4], ref_arr_Data.shape[1])

    np.savetxt(data_dir + demons_nums[d] + "_sample_frame1_b.txt", temp_arr_frame1_b, fmt='%.5f')
    np.savetxt(data_dir + demons_nums[d] + "_sample_frame1_A.txt", temp_arr_frame1_A, fmt='%.5f')
    np.savetxt(data_dir + demons_nums[d] + "_sample_frame2_b.txt", temp_arr_frame2_b, fmt='%.5f')
    np.savetxt(data_dir + demons_nums[d] + "_sample_frame2_A.txt", temp_arr_frame2_A, fmt='%.5f')

    with open(r"{}".format(data_dir) + demons_nums[d] + "_sample_frame1_b.txt", 'r') as f:
        data = f.read(); data = data.replace(' ', ',')
    with open(r"{}".format(data_dir) + demons_nums[d] + "_sample_frame1_b.txt", 'w') as f:
        f.write(data)

    with open(r"{}".format(data_dir) + demons_nums[d] + "_sample_frame1_A.txt", 'r') as f:
        data = f.read(); data = data.replace(' ', ',')
    with open(r"{}".format(data_dir) + demons_nums[d] + "_sample_frame1_A.txt", 'w') as f:
        f.write(data)

    with open(r"{}".format(data_dir) + demons_nums[d] + "_sample_frame2_b.txt", 'r') as f:
        data = f.read(); data = data.replace(' ', ',')
    with open(r"{}".format(data_dir) + demons_nums[d] + "_sample_frame2_b.txt", 'w') as f:
        f.write(data)

    with open(r"{}".format(data_dir) + demons_nums[d] + "_sample_frame2_A.txt", 'r') as f:
        data = f.read(); data = data.replace(' ', ',')
    with open(r"{}".format(data_dir) + demons_nums[d]+ "_sample_frame2_A.txt", 'w') as f:
        f.write(data)

    ## Only for debuggin to check the txt file shape
    print("Debugging")
    temp_arr_Data = np.loadtxt(data_dir + demons_nums[d] + "_sample_Data.txt", delimiter=',')
    print(temp_arr_Data.shape)
    print("\n")

    demons_nums_dtw.append(demons_nums[d])
    
# Resend the dictionary again after filtering the Demons according to DTW (i.e. keeping the most similar ones)
ref_demon["demons_nums"] = demons_nums_dtw
ref_demon["nbDemons"] = len(ref_demon["demons_nums"])
print("Reference Demon:")
print(ref_demon)
with open(tasks_dir / f'{task_name}/demons_info.pkl', 'wb') as fp:
    pickle.dump(ref_demon, fp)    


In [ ]:
# print(temp_arr_frame1_b.shape)
# print(temp_arr_frame1_A.shape)
# print(temp_arr_frame2_b.shape)
# print(temp_arr_frame2_A.shape)
# print(temp_arr_frame2_A[:4,:4].shape)